# Baselines

Standard GCN, GAT, and GraphSAGE models on Elliptic++.

In [ ]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, recall_score, roc_auc_score, precision_recall_curve, auc
import torch_geometric.transforms as T
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
DATA_DIR = '../data/elliptic++/'


/home/ankit/fraud_detection/venv/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /home/ankit/fraud_detection/venv/lib/python3.10/site-packages/libpyg.so
  import torch_geometric.typing
/home/ankit/fraud_detection/venv/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /home/ankit/fraud_detection/venv/lib/python3.10/site-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/home/ankit/fraud_detection/venv/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /home/ankit/fraud_detection/venv/lib/python3.10/site-packages/torch_sparse/_version_cuda.so
  import torch_geometric.typi

Using device: cuda


/home/ankit/fraud_detection/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load features, classes, and edges
print("Loading data...")
wallet_features = pd.read_csv(DATA_DIR + 'wallets_features.csv')
wallet_classes = pd.read_csv(DATA_DIR + 'wallets_classes.csv')
edges = pd.read_csv(DATA_DIR + 'AddrAddr_edgelist.csv')

# Unique addresses and their mapping to integer IDs (0 to N-1)
# wallet_features contains all addresses. Wait, there can be duplicate addresses across time steps? 
# EDA showed addresses can appear multiple times if they are active in multiple time steps. 
# but features can duplicate. Let's merge on address and take the first occurrence or specific time step
wallet_features = wallet_features.drop_duplicates(subset='address', keep='first')
wallet_merged = wallet_features.merge(wallet_classes, on='address', how='left')

# Mapping addresses to node indices
addresses = wallet_merged['address'].values
addr_to_idx = {addr: i for i, addr in enumerate(addresses)}

# Build feature matrix X
# features are from column 2 to end (skip address and Time step)
feature_cols = [c for c in wallet_merged.columns if c not in ['address', 'Time step', 'class']]
X = torch.tensor(wallet_merged[feature_cols].values, dtype=torch.float32)

# Build labels Y (Illicit = 1 (label 1), Licit = 2 (label 0), Unknown = 3 (label -1))
class_vals = wallet_merged['class'].fillna(3).values
y_mapped = np.zeros_like(class_vals, dtype=np.int64) - 1
y_mapped[class_vals == 1] = 1 # Illicit
y_mapped[class_vals == 2] = 0 # Licit
Y = torch.tensor(y_mapped, dtype=torch.long)

# Build edge_index
edge_src = [addr_to_idx[src] for src in edges['input_address'] if src in addr_to_idx]
edge_dst = [addr_to_idx[dst] for dst in edges['output_address'] if dst in addr_to_idx]
edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)

# Time steps for splitting
time_steps = wallet_merged['Time step'].values

# Elliptic standard split: 
# Train: 1 to 31
# Val: 32 to 34
# Test: 35 to 49
train_mask = (time_steps <= 31) & (y_mapped != -1)
val_mask = ((time_steps >= 32) & (time_steps <= 34)) & (y_mapped != -1)
test_mask = (time_steps >= 35) & (y_mapped != -1)

data = Data(x=X, edge_index=edge_index, y=Y)
data.train_mask = torch.tensor(train_mask, dtype=torch.bool)
data.val_mask = torch.tensor(val_mask, dtype=torch.bool)
data.test_mask = torch.tensor(test_mask, dtype=torch.bool)

# Since baseline MPNNs are UNDIRECTED, make graph undirected
data = T.ToUndirected()(data)

print(data)
print(f"Train nodes: {data.train_mask.sum().item()}")
print(f"Val nodes:   {data.val_mask.sum().item()}")
print(f"Test nodes:  {data.test_mask.sum().item()}")

# Illicit class weight
train_y = data.y[data.train_mask]
n_licit = (train_y == 0).sum().item()
n_illicit = (train_y == 1).sum().item()
illicit_weight = n_licit / max(n_illicit, 1)
class_weights = torch.tensor([1.0, illicit_weight], dtype=torch.float32).to(device)
print(f"Computed Class Weights: Licit=1.0, Illicit={illicit_weight:.2f}")

data = data.to(device)


Loading data...
Data(x=[822942, 55], edge_index=[2, 5553655], y=[822942], train_mask=[822942], val_mask=[822942], test_mask=[822942])
Train nodes: 159414
Val nodes:   13489
Test nodes:  92451
Computed Class Weights: Licit=1.0, Illicit=19.19


In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        # Reduce heads to 2 to save memory
        self.conv1 = GATConv(in_channels, hidden_channels, heads=2, concat=False)
        self.conv2 = GATConv(hidden_channels, out_channels, heads=1, concat=False)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

In [ ]:
def train(model, optimizer, data):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], weight=class_weights)
    loss.backward()
    optimizer.step()
    return loss.item()

def test(model, data, mask):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        probs = F.softmax(out[mask], dim=1)
        preds = out[mask].argmax(dim=1).cpu().numpy()
        probs_illicit = probs[:, 1].cpu().numpy()
        y_true = data.y[mask].cpu().numpy()
        
        f1 = f1_score(y_true, preds, pos_label=1, zero_division=0)
        f1_micro = f1_score(y_true, preds, average='micro', zero_division=0)
        recall = recall_score(y_true, preds, pos_label=1, zero_division=0)
        try:
            auc_test = roc_auc_score(y_true, probs_illicit)
            precision, recall_curve_vals, _ = precision_recall_curve(y_true, probs_illicit)
            auprc = auc(recall_curve_vals, precision)
        except ValueError:
            auc_test = 0.0
            auprc = 0.0
            
    return f1, f1_micro, recall, auc_test, auprc

def run_experiment(model_class, data, epochs=100, lr=0.01, hidden=128):
    print(f"\n--- Running {model_class.__name__} ---")
    torch.cuda.empty_cache()
    if model_class.__name__ == 'GAT':
        hidden = 64
    model = model_class(data.x.shape[1], hidden, 2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    
    best_val_f1 = 0
    best_test_metrics = None
    
    for epoch in range(1, epochs + 1):
        loss = train(model, optimizer, data)
        val_f1, val_f1_micro, val_rec, val_auc, val_auprc = test(model, data, data.val_mask)
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_test_metrics = test(model, data, data.test_mask)
            
        if epoch % 20 == 0:
            print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val F1: {val_f1:.4f}')
            
    test_f1, test_f1_micro, test_rec, test_auc, test_auprc = best_test_metrics
    print(f"\nBest Test metrics based on Val F1:")
    print(f"F1 (Illicit): {test_f1:.4f} | Micro-F1: {test_f1_micro:.4f} | Recall: {test_rec:.4f} | AUC: {test_auc:.4f} | AUPRC: {test_auprc:.4f}")
    return best_test_metrics

In [ ]:
# Run models
results = {}
epochs = 200

results['GCN'] = run_experiment(GCN, data, epochs=epochs)
results['GAT'] = run_experiment(GAT, data, epochs=epochs)
results['GraphSAGE'] = run_experiment(GraphSAGE, data, epochs=epochs)

# Display Summary
print("\n" + "="*50)
print("FINAL BASELINE RESULTS (Minority Class: Illicit)")
print("="*50)
df_res = pd.DataFrame(results, index=['F1 (Illicit)', 'Micro-F1', 'Recall', 'AUC-ROC', 'AUPRC']).T
print(df_res.round(4))


--- Running GCN ---
Epoch: 020, Loss: 6231.5894, Val F1: 0.4621
Epoch: 040, Loss: 2687.9993, Val F1: 0.5324
Epoch: 060, Loss: 901.2673, Val F1: 0.4438
Epoch: 080, Loss: 161.3297, Val F1: 0.4760
Epoch: 100, Loss: 193.6407, Val F1: 0.2339
Epoch: 120, Loss: 33.7257, Val F1: 0.2340
Epoch: 140, Loss: 49.7063, Val F1: 0.2612
Epoch: 160, Loss: 58.2825, Val F1: 0.4796
Epoch: 180, Loss: 51.9222, Val F1: 0.2584
Epoch: 200, Loss: 30.9331, Val F1: 0.4415

Best Test metrics based on Val F1:
F1 (Illicit): 0.3158 | Micro-F1: 0.9246 | Recall: 0.3289 | AUC: 0.7724 | AUPRC: 0.2957

--- Running GAT ---
Epoch: 020, Loss: 4826.9414, Val F1: 0.4465
Epoch: 040, Loss: 2167.6562, Val F1: 0.4468
Epoch: 060, Loss: 408.5630, Val F1: 0.4439
Epoch: 080, Loss: 54.7495, Val F1: 0.2214
Epoch: 100, Loss: 92.4657, Val F1: 0.0000
Epoch: 120, Loss: 92.3038, Val F1: 0.2321
Epoch: 140, Loss: 55.0461, Val F1: 0.2209
Epoch: 160, Loss: 17.3139, Val F1: 0.4419
Epoch: 180, Loss: 62.3355, Val F1: 0.4267
Epoch: 200, Loss: 20.3775